# 05 — Final Load Prep (KPIs)

The cleaned dataset (`data/processed/amazon_products_clean.csv`) is the single source of truth for both the report and the Tableau dashboard — no separate extract is maintained.

This notebook computes the KPI summary table consumed by the report and dashboard:
- `data/processed/amazon_kpis.csv` — single source for the report's KPI section.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "processed" / "amazon_products_clean.csv")
print(df.shape)
df.head(2)

(5971, 21)


,title,rating,number_of_reviews,bought_in_last_month,current_discounted_price,listed_price,is_best_seller,is_sponsored,collected_at,has_buy_box,...,has_coupon,has_sustainability_badge,selling_price,discount_pct,is_discounted,brand,price_tier,rating_bucket,log_reviews,log_bought_last_month
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300.0,89.68,159.00,False,True,2025-08-21 11:14:29,True,...,True,True,89.68,43.60,True,Boya,Premium,Excellent,5.929589,5.707110
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000.0,9.99,15.99,False,True,2025-08-21 11:14:29,True,...,False,False,9.99,37.52,True,Lisen,Budget,Good,7.807103,8.699681


## 1. KPI definitions

| KPI | Formula | Why it matters |
|---|---|---|
| Catalogue Size | `count(rows)` | Inventory scope |
| Avg / Median Selling Price | `mean / median(selling_price)` | Pricing positioning |
| Weighted Avg Rating | `sum(rating × reviews) / sum(reviews)` | Real customer voice (weights stronger SKUs) |
| Total Reviews | `sum(number_of_reviews)` | Total catalogue trust signal |
| % Discounted | `mean(is_discounted)` | Promo aggressiveness |
| Avg Discount when on sale | `mean(discount_pct \| discount > 0)` | Discount depth |
| % Sponsored | `mean(is_sponsored)` | Ad-spend share |
| % Best Seller | `mean(is_best_seller)` | Catalogue strength |
| % Buy Box available | `mean(has_buy_box)` | Listing health |

In [2]:
kpis = {}
kpis["catalogue_size"] = len(df)
kpis["avg_selling_price_usd"] = round(df["selling_price"].mean(), 2)
kpis["median_selling_price_usd"] = round(df["selling_price"].median(), 2)
w = df["number_of_reviews"].fillna(0).clip(lower=1)
kpis["weighted_avg_rating"] = round(np.average(df["rating"].fillna(df["rating"].mean()), weights=w), 3)
kpis["avg_rating"] = round(df["rating"].mean(), 3)
kpis["total_reviews"] = int(df["number_of_reviews"].fillna(0).sum())
kpis["pct_discounted"] = round(df["is_discounted"].mean() * 100, 2)
kpis["avg_discount_when_on_sale"] = round(df.loc[df["is_discounted"], "discount_pct"].mean(), 2)
kpis["pct_sponsored"] = round(df["is_sponsored"].mean() * 100, 2)
kpis["pct_best_seller"] = round(df["is_best_seller"].mean() * 100, 2)
kpis["pct_with_coupon"] = round(df["has_coupon"].mean() * 100, 2)
kpis["pct_buy_box"] = round(df["has_buy_box"].mean() * 100, 2)
kpis["unique_brands_heuristic"] = df["brand"].nunique()
top_brand = df["brand"].value_counts().head(1)
kpis["top_brand"] = top_brand.index[0]
kpis["top_brand_share_pct"] = round(top_brand.iloc[0] / len(df) * 100, 2)

kpi_df = pd.DataFrame({"kpi": list(kpis.keys()), "value": list(kpis.values())})
kpi_df

,kpi,value
0,catalogue_size,5971
1,avg_selling_price_usd,180.81
2,median_selling_price_usd,71.95
3,weighted_avg_rating,4.578
4,avg_rating,4.453
5,total_reviews,37901614
6,pct_discounted,46.12
7,avg_discount_when_on_sale,21.09
8,pct_sponsored,4.69
9,pct_best_seller,2.68


## 2. Write outputs

Tableau reads `amazon_products_clean.csv` directly. Only the KPI table is written here.

In [3]:
out_dir = ROOT / "data" / "processed"
kpi_path = out_dir / "amazon_kpis.csv"
kpi_df.to_csv(kpi_path, index=False)
print("Wrote:", kpi_path)
print("Tableau source:", out_dir / "amazon_products_clean.csv", "(no separate extract)")

Wrote: /Users/aashishjha/Documents/Projects/DVA-capstone-2/data/processed/amazon_kpis.csv
Tableau source: /Users/aashishjha/Documents/Projects/DVA-capstone-2/data/processed/amazon_products_clean.csv (no separate extract)


## 3. Hand-off checklist

- [ ] `data/processed/amazon_products_clean.csv` ✓ (5,971 × 21, zero NaN)
- [ ] `data/processed/amazon_kpis.csv` ✓
- [ ] Tableau workbook reads from `amazon_products_clean.csv`
- [ ] Tableau Public URL pasted into `tableau/dashboard_links.md`